## Loading dependencies

In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes

In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.9 MB/s eta 0:00:00


## loading data for finetuning

In [1]:
from pypdf import PdfReader
file = PdfReader("/content/ml_interview_500_questions_answers.pdf")
text = ""
for page in file.pages:
  text+=page.extract_text()
print(text[:100])

Q1. [Machine Learning] What is overfitting?
A1. Overfitting occurs when a model memorizes training d


In [2]:
len(text)

63964

In [3]:
lines = text.split("\n")

In [4]:
lines[5]

'Q3. [Linear Algebra] What is a matrix?'

In [7]:
type(text)

str

## Formatting data

In [5]:
import re
dataset = []
for i in range(len(lines)):
  if(lines[i].startswith("Q")):
    lines[i] = re.sub(r'Q\d+', '', lines[i])
    lines[i] = re.sub(r'\[.*?\]', '', lines[i])
    question = lines[i]
  elif(lines[i].startswith("A")):
    lines[i] = re.sub(r'A\d+', '', lines[i])
    lines[i] = re.sub(r'\[.*?\]', '', lines[i])
    output = lines[i]
    result = {'instruction': question,'output' : output}
    dataset.append(result)

In [6]:
dataset[0]

{'instruction': '.  What is overfitting?',
 'output': '. Overfitting occurs when a model memorizes training data and performs poorly on unseen data.'}

In [7]:
import torch

# 1. Define the device based on CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Check the specific GPU name assigned to you
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


##Downloading Phi 3.8B model from unsloth

In [9]:
from unsloth import FastLanguageModel
model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
max_seq_length = 2048
dtype = None
model,tokenizer = FastLanguageModel.from_pretrained(model_name = model_name,max_seq_length=max_seq_length,dtype=dtype,load_in_4bit=True)

==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

In [12]:
from datasets import Dataset

def format_prompt(example):
    return f"### Input: {example['instruction']}\n### Output: {(example['output'])}<|endoftext|>"

formatted_data = [format_prompt(item) for item in dataset]
dataset = Dataset.from_dict({"text": formatted_data})

In [13]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

In [16]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt
messages = [
    {"role": "user", "content": "What is matrix multiplication"}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|> What is matrix multiplication<|end|><|assistant|> Matrix multiplication is a binary operation that takes two matrices, and produces another matrix. This operation is not commutative, which means the order in compositions matters. The product of two matrices A and B is a new matrix C, where each element c_ij of C is calculated by taking the dot product of the i-th row of A and the j-th column of B.


Mathematically, if A is an m×n matrix and B is an n×p matrix, their product AB will be an m×p matrix C. The element c_ij in C is given by:


c_ij = a_i1*b_1j + a_i2*b_2j + ... + a_in*b_nj


where a_ik is the element in the i-th row and k-th column of matrix A, and b_kj is the element in the k-th row and j-th column of matrix B.


It is important to note that for two matrices to be multipliable, the number of columns in the first matrix must be equal to the number of rows in the second matrix. If A is an m×n


##loading inference data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
finetuned_model_path = "/content/drive/MyDrive/my_finetuned_model"
model.save_pretrained(finetuned_model_path)
tokenizer.save_pretrained(finetuned_model_path)

print(f"Fine-tuned model and tokenizer saved to: {finetuned_model_path}")

In [30]:
baseline_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
finetuned_name = "/content/drive/MyDrive/my_finetuned_model"

In [20]:
inference_example = PdfReader("/content/100_unique_ml_questions.pdf")
text = ""
for page in inference_example.pages:
  text+=page.extract_text()
print(text[:100])

Q1. What is gradient descent?
A1. Gradient descent is an optimization algorithm that minimizes loss 


In [24]:
inf_lines = text.split("\n")

In [25]:
infdataset = []
for i in range(len(inf_lines)):
  if(inf_lines[i].startswith("Q")):
    inf_lines[i] = re.sub(r'Q\d+', '', inf_lines[i])
    question = inf_lines[i]
  elif(inf_lines[i].startswith("A")):
    inf_lines[i] = re.sub(r'A\d+', '', inf_lines[i])
    output = inf_lines[i]
    result = {'instruction': question,'output' : output}
    infdataset.append(result)

In [27]:
infdataset[0]

{'instruction': '. What is gradient descent?',
 'output': '. Gradient descent is an optimization algorithm that minimizes loss by moving parameters'}

In [28]:
formatted_data = [format_prompt(item) for item in infdataset]
infdataset = Dataset.from_dict({"text": formatted_data})

In [29]:
infdataset[0]

{'text': '### Input: . What is gradient descent?\n### Output: . Gradient descent is an optimization algorithm that minimizes loss by moving parameters<|endoftext|>'}

### Using the Baseline Model for Answering Questions

To see how the original `unsloth/Phi-3-mini-4k-instruct-bnb-4bit` model (your baseline) performs, we'll load it and then use it to answer a question. This will allow for a direct comparison with your fine-tuned model's performance.

In [31]:
# Load the baseline model and tokenizer
from unsloth import FastLanguageModel

baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name=baseline_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(baseline_model)

print(f"Baseline model loaded from {baseline_name}")

==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Baseline model loaded from unsloth/Phi-3-mini-4k-instruct-bnb-4bit


Now, let's create a function to generate answers using this baseline model.

In [32]:
def generate_baseline_answer(instruction_text, model, tokenizer):
    messages = [
        {"role": "user", "content": instruction_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_tag = "<|assistant|>"
    if assistant_tag in decoded_output:
        return decoded_output.split(assistant_tag, 1)[1].strip()
    return decoded_output


In [33]:
def generate_answer(instruction_text, model, tokenizer):
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "user", "content": instruction_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)
    decoded_output1 = tokenizer.decode(outputs[0], skip_special_tokens=True)
    assistant_tag = "<|assistant|>"
    if assistant_tag in decoded_output1:
      return decoded_output1.split(assistant_tag, 1)[1].strip()
    return decoded_output1


In [38]:
instruction_to_pass = infdataset[0]['text'].split('### Input: ')[1].split('\n### Output:')[0].strip()
print(generate_baseline_answer(instruction_to_pass, baseline_model, baseline_tokenizer))

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


. What is gradient descent? Gradient descent is an optimization algorithm used to minimize some function by iteratively moving in the direction of the steepest descent as defined by the negative of the gradient. In machine learning, it is used to update the parameters of a model. Parameters refer to the coefficients or weights, which are adjusted to minimize a loss function that measures how well the model is performing. The goal of gradient descent is to find the best parameters that result in the lowest possible value of the loss function, which corresponds to the most accurate model.


##Comparison of baseline model and finetuned model

In [43]:
question = infdataset[0]
instruction_to_pass = question['text'].split('### Input: ')[1].split('\n### Output:')[0].strip()
base_answer = generate_baseline_answer(instruction_to_pass, baseline_model, baseline_tokenizer)
finetuned_answer = generate_answer(instruction_to_pass, model, tokenizer)
print('\n')
print(f"Question: {instruction_to_pass}")
print('\n')
print(f"Baseline Answer: {base_answer}")
print('\n')
print(f"Fine-Tuned Answer: {finetuned_answer}")
print('\n')

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Question: . What is gradient descent?


Baseline Answer: . What is gradient descent? Gradient descent is an optimization algorithm used to minimize a function by iteratively moving in the direction of the steepest descent as defined by the negative of the gradient. In machine learning, it is used to update the parameters of a model in order to minimize a cost function.


Fine-Tuned Answer: . What is gradient descent? Gradient descent is an optimization algorithm used in machine learning and statistics to minimize a function. In the context of machine learning, it'ies used to find the values of the parameters of a function that minimize a cost function, which measures how far the predictions of a model are from the actual results.


The basic idea of gradient descent is to iteratively adjust the parameters of the model in the direction of the steepest descent of the cost function. This direction is given by the negative of the gradient of the cost function at the current point. By rep

In [ ]:
all_baseline_predictions = []
all_finetuned_predictions = []

for question_item in infdataset:
  instruction_to_pass = question_item['text'].split('### Input: ')[1].split('\n### Output:')[0].strip()

  base_answer = generate_baseline_answer(instruction_to_pass, baseline_model, baseline_tokenizer)
  all_baseline_predictions.append(base_answer)

  finetuned_answer = generate_answer(instruction_to_pass, model, tokenizer)
  all_finetuned_predictions.append(finetuned_answer)

print(f"Collected {len(all_baseline_predictions)} answers from the baseline model.")
print(f"Collected {len(all_finetuned_predictions)} answers from the fine-tuned model.")

In [ ]:
#stopped for time,compute and memory reasons

In [50]:
len(all_baseline_predictions)

80

### Initialize Gemini API for LLM-as-a-Judge Evaluation

To use Gemini as a judge, we first need to import the SDK and configure it with your API key. Make sure your `GOOGLE_API_KEY` is stored in Colab secrets.

In [55]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Generative Model for judging
llm_judge_model = genai.GenerativeModel('gemini-3.5-flash') # Using a capable model for judging
print("Gemini LLM judge model initialized.")

Gemini LLM judge model initialized.


In [56]:
def evaluate_with_llm_judge(
    question: str,
    ground_truth_answer: str,
    baseline_model_answer: str,
    finetuned_model_answer: str,
    judge_model
):
    prompt = f"""
    You are an impartial judge evaluating the quality of answers provided by two AI models (Baseline and Finetuned) to a given question.
    You will also be provided with the Ground Truth answer for the question.

    Your task is to:
    1. Carefully read the Question.
    2. Carefully read the Ground Truth Answer.
    3. Carefully read the Baseline Model's Answer.
    4. Carefully read the Finetuned Model's Answer.
    5. Evaluate each model's answer based on correctness, relevance, completeness, and conciseness relative to the Ground Truth.
    6. Provide a score for the Baseline Model (1-5, where 5 is excellent and 1 is very poor).
    7. Provide a score for the Finetuned Model (1-5, where 5 is excellent and 1 is very poor).
    8. Provide a brief explanation for each score, highlighting strengths and weaknesses.

    Question: {question}

    Ground Truth Answer: {ground_truth_answer}

    Baseline Model Answer: {baseline_model_answer}

    Finetuned Model Answer: {finetuned_model_answer}

    Please format your response as follows:

    ## Evaluation Results

    ### Baseline Model Score: [SCORE/5]
    Explanation: [Explanation for Baseline Model]

    ### Finetuned Model Score: [SCORE/5]
    Explanation: [Explanation for Finetuned Model]
    """

    try:
        response = judge_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error during evaluation: {e}"

print("LLM judge evaluation function defined.")

LLM judge evaluation function defined.


In [57]:
# Take a small subset to demonstrate to avoid too many API calls
num_eval_samples = 5 # You can increase this to evaluate more samples

llm_judge_results = []

for i in range(min(num_eval_samples, len(infdataset))):
    question_item = infdataset[i]

    # Extract question and ground truth from the formatted text
    # The 'text' field is '### Input: QUESTION\n### Output: GROUND_TRUTH<|endoftext|>' format
    parts = question_item['text'].split('### Input: ')[1].split('\n### Output:')
    question = parts[0].strip()
    ground_truth_answer = parts[1].replace('<|endoftext|>', '').strip()

    baseline_ans = all_baseline_predictions[i]
    finetuned_ans = all_finetuned_predictions[i]

    print(f"\n--- Evaluating Sample {i+1} ---")
    print(f"Question: {question}")

    evaluation = evaluate_with_llm_judge(
        question=question,
        ground_truth_answer=ground_truth_answer,
        baseline_model_answer=baseline_ans,
        finetuned_model_answer=finetuned_ans,
        judge_model=llm_judge_model
    )
    llm_judge_results.append(evaluation)
    print(evaluation)

print("\nLLM-as-a-judge evaluation complete for selected samples.")


--- Evaluating Sample 1 ---
Question: . What is gradient descent?
## Evaluation Results

### Baseline Model Score: 3.5/5
**Explanation**: The Baseline Model provides a correct and detailed explanation of gradient descent, capturing the essence of the ground truth (optimization algorithm, minimizing loss/cost, updating parameters). However, it attempts to provide a Python code example which is cut off mid-sentence, contains a typo (`npnant`), and remains incomplete. This hurts the overall quality and conciseness of the response.

### Finetuned Model Score: 4.8/5
**Explanation**: The Finetuned Model provides an excellent, clear, and accurate explanation of gradient descent that aligns perfectly with the ground truth while providing the necessary context. It explains the concept of "steepest descent" and how it relates to updating model parameters to minimize loss. There is a minor formatting/generation artifact at the very end of the response (the character "串"), but the actual content 